In [1]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
import pandas as pd

# Load your already extracted landmarks
df = pd.read_csv("/content/drive/MyDrive/landmarks_dataset.csv")
df = df[df['label'] != 'nothing']  # drop nothing class
print(f" Loaded {len(df)} samples")

 Loaded 69273 samples


In [3]:
import numpy as np

def normalize_landmarks(row):
    # row is a list of 63 numbers (21 landmarks × x,y,z)
    landmarks = np.array(row).reshape(21, 3)

    # Step 1 - Subtract wrist (landmark 0) to center the hand
    wrist = landmarks[0]
    landmarks = landmarks - wrist

    # Step 2 - Scale by hand size (distance from wrist to middle finger tip = landmark 12)
    hand_size = np.linalg.norm(landmarks[12])
    if hand_size > 0:
        landmarks = landmarks / hand_size

    return landmarks.flatten().tolist()

# Apply to the dataset
df_normalized = df.copy()
feature_cols = [col for col in df.columns if col != 'label']
df_normalized[feature_cols] = df[feature_cols].apply(normalize_landmarks, axis=1, result_type='expand')

print(" Normalization done!")
print(df_normalized.head())

 Normalization done!
   x_0  y_0  z_0       x_1       y_1       z_1       x_2       y_2       z_2  \
0  0.0  0.0  0.0  0.477782  0.127621 -0.222858  0.995142 -0.241571 -0.256410   
1  0.0  0.0  0.0  0.491615  0.100576 -0.237233  0.914569 -0.091256 -0.357225   
2  0.0  0.0  0.0  0.513145  0.066206 -0.399852  0.893397 -0.188898 -0.572792   
3  0.0  0.0  0.0  0.409945  0.175293 -0.121234  0.858529 -0.028685 -0.165521   
4  0.0  0.0  0.0  0.458869  0.109846 -0.092339  0.891527 -0.131039 -0.108662   

        x_3  ...      x_18      y_18      z_18      x_19      y_19      z_19  \
0  0.936041  ...  0.214827 -1.249232 -0.515912  0.198139 -0.836007 -0.503450   
1  0.965158  ...  0.296930 -0.990617 -0.514774  0.281940 -0.672243 -0.543883   
2  0.859118  ...  0.314031 -1.281348 -0.496726  0.210787 -0.956719 -0.503319   
3  0.931711  ...  0.350996 -0.984950 -0.508471  0.290525 -0.708491 -0.504216   
4  0.934477  ...  0.342746 -1.010406 -0.314895  0.345302 -0.756010 -0.324456   

       x_20      

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Features and labels
feature_cols = [col for col in df_normalized.columns if col != 'label']
X = df_normalized[feature_cols].values
y = df_normalized['label'].values

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Train
clf_normalized = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
clf_normalized.fit(X_train, y_train)

# Evaluate
y_pred = clf_normalized.predict(X_test)
print(f"Normalized RF Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"Previous accuracy: 97.64%")

Normalized RF Accuracy: 98.17%
Previous accuracy: 97.64%


In [5]:
# Install xgboost
!pip install xgboost

from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

# XGBoost needs numeric labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# Train
xgb = XGBClassifier(n_estimators=100, random_state=42, n_jobs=-1, use_label_encoder=False, eval_metric='mlogloss')
xgb.fit(X_train, y_train_enc)

# Evaluate
y_pred_xgb = xgb.predict(X_test)
y_pred_xgb_labels = le.inverse_transform(y_pred_xgb)
print(f" XGBoost Accuracy: {accuracy_score(y_test, y_pred_xgb_labels) * 100:.2f}%")

/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [14:47:13] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


 XGBoost Accuracy: 98.43%


In [6]:
import joblib
joblib.dump(xgb, "/content/drive/MyDrive/asl_model_xgb.pkl")
joblib.dump(le, "/content/drive/MyDrive/label_encoder.pkl")
print(" XGBoost model saved!")

 XGBoost model saved!


In [7]:
import joblib
joblib.dump(xgb, "/content/drive/MyDrive/asl_model_xgb.pkl")
joblib.dump(le, "/content/drive/MyDrive/label_encoder.pkl")
print("Saved!")

Saved!


In [8]:
import os
print(f"XGBoost model: {os.path.getsize('/content/drive/MyDrive/asl_model_xgb.pkl') / 1024 / 1024:.2f} MB")
print(f"Label encoder: {os.path.getsize('/content/drive/MyDrive/label_encoder.pkl') / 1024 / 1024:.2f} MB")

XGBoost model: 3.95 MB
Label encoder: 0.00 MB
